# NAICS-2 Cross-Encoder — (DeBERTa-v3-base)

In [1]:
!pip install transformers datasets accelerate scikit-learn sentencepiece protobuf -q

In [2]:
from google.colab import drive
import os, zipfile

drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/deberta-v3-base.zip"
cache_dir = os.path.expanduser("~/.cache/huggingface/hub/")
os.makedirs(cache_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    for info in z.infolist():
        fixed_name = info.filename.replace("\\", "/")
        out_path = os.path.join(cache_dir, fixed_name)
        if info.is_dir() or fixed_name.endswith("/"):
            os.makedirs(out_path, exist_ok=True)
        elif info.file_size == 0:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
        else:
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            with z.open(info) as src, open(out_path, 'wb') as dst:
                dst.write(src.read())

model_cache = os.path.join(cache_dir, "models--microsoft--deberta-v3-base")
print(f"Model extracted to: {model_cache}")
print(f"Contents: {os.listdir(model_cache)}")

Mounted at /content/drive
Model extracted to: /root/.cache/huggingface/hub/models--microsoft--deberta-v3-base
Contents: ['refs', 'snapshots', 'blobs']


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import json
import os
import gc
import re
import random
import shutil

os.environ["HF_HUB_OFFLINE"] = "1"

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [4]:
df_raw = pd.read_csv('ExioNAICS.csv')

SECTOR_MERGE = {
    '31': '31-33', '32': '31-33', '33': '31-33',
    '44': '44-45', '45': '44-45',
    '48': '48-49', '49': '48-49',
}

naics2_raw = df_raw[['NAICS_2 Code', 'NAICS_2 Title', 'NAICS_2 Description']].drop_duplicates(subset='NAICS_2 Code').copy()
naics2_raw['NAICS_2 Code'] = naics2_raw['NAICS_2 Code'].astype(str)
naics2_raw['sector_code'] = naics2_raw['NAICS_2 Code'].map(SECTOR_MERGE).fillna(naics2_raw['NAICS_2 Code'])

naics2_corpus = naics2_raw.drop_duplicates(subset='sector_code').copy()
naics2_corpus = naics2_corpus.sort_values('sector_code').reset_index(drop=True)

sector_codes  = naics2_corpus['sector_code'].tolist()
sector_titles = naics2_corpus['NAICS_2 Title'].tolist()
code_to_idx   = {code: i for i, code in enumerate(sector_codes)}
NUM_CLASSES   = len(sector_codes)

df = pd.read_csv('ExioNAICS_preprocessed.csv')
df['NAICS Code'] = df['NAICS Code'].astype(str)
df['naics2_raw'] = df['NAICS Code'].str[:2]
df['sector'] = df['naics2_raw'].map(SECTOR_MERGE).fillna(df['naics2_raw'])
df['naics2_idx'] = df['sector'].map(code_to_idx)
df = df.dropna(subset=['naics2_idx']).reset_index(drop=True)
df['naics2_idx'] = df['naics2_idx'].astype(int)

print(f"NAICS-2 sectors: {NUM_CLASSES}")
print(f"Dataset: {len(df)} samples")


def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


print("\n=== Building NAICS-6 augmentation training data ===")
naics_sources = [
    ('NAICS-6', 'NAICS Code', 'NAICS Title', 'Description'),
    ('NAICS-5', 'NAICS_5 Code', 'NAICS_5 Title', 'NAICS_5 Description'),
    ('NAICS-4', 'NAICS_4 Code', 'NAICS_4 Title', 'NAICS_4 Description'),
    ('NAICS-3', 'NAICS_3 Code', 'NAICS_3 Title', 'NAICS_3 Description'),
]

aug_rows = []
for level, code_col, title_col, desc_col in naics_sources:
    sub = df_raw[[code_col, title_col, desc_col]].drop_duplicates(subset=code_col).dropna(subset=[code_col])
    for _, row in sub.iterrows():
        code = str(row[code_col])
        title = str(row[title_col]) if not pd.isna(row[title_col]) else ""
        desc  = str(row[desc_col])  if not pd.isna(row[desc_col])  else ""
        text = (title + ". " + desc).strip()
        if len(text) < 20:
            continue
        naics2_prefix = code[:2]
        sector = SECTOR_MERGE.get(naics2_prefix, naics2_prefix)
        if sector not in code_to_idx:
            continue
        aug_rows.append({
            'naics_code': code,
            'level': level,
            'clean_description': preprocess_text(text),
            'sector': sector,
            'naics2_idx': code_to_idx[sector],
        })

df_aug = pd.DataFrame(aug_rows).drop_duplicates(subset='naics_code').reset_index(drop=True)
print(f"Augmentation samples: {len(df_aug)}")


print("\n=== Enriching NAICS-2 corpus with NAICS-6 example titles ===")
MAX_EXAMPLES_PER_SECTOR = 25

naics6_titles = df_raw[['NAICS Code', 'NAICS Title']].drop_duplicates(subset='NAICS Code').dropna()
naics6_titles['NAICS Code'] = naics6_titles['NAICS Code'].astype(str)
naics6_titles['sector'] = naics6_titles['NAICS Code'].str[:2].map(SECTOR_MERGE).fillna(naics6_titles['NAICS Code'].str[:2])

sector_to_examples = {}
for sector, grp in naics6_titles.groupby('sector'):
    titles = [t.strip() for t in grp['NAICS Title'].astype(str).tolist() if t.strip()]
    sector_to_examples[sector] = titles[:MAX_EXAMPLES_PER_SECTOR]

corpus_texts = []
for i, code in enumerate(sector_codes):
    title = sector_titles[i]
    description = naics2_corpus['NAICS_2 Description'].iloc[i]
    description = str(description) if pd.notna(description) else ""
    examples = sector_to_examples.get(code, [])
    if examples:
        text = f"{title}. Examples: {'; '.join(examples)}. {description}"
    else:
        text = f"{title}. {description}"
    corpus_texts.append(text)

print(f"Enriched corpus length (chars): "
      f"min={min(len(t) for t in corpus_texts)}, "
      f"max={max(len(t) for t in corpus_texts)}, "
      f"mean={int(np.mean([len(t) for t in corpus_texts]))}")

del df_raw
gc.collect()

NAICS-2 sectors: 20
Dataset: 20535 samples

=== Building NAICS-6 augmentation training data ===
Augmentation samples: 2173

=== Enriching NAICS-2 corpus with NAICS-6 example titles ===
Enriched corpus length (chars): min=1141, max=8357, mean=3365


90

In [5]:
MODEL_NAME       = "microsoft/deberta-v3-base"
MAX_LENGTH       = 192
BATCH_SIZE       = 8
NUM_EPOCHS       = 12
LEARNING_RATE    = 1.5e-5
WARMUP_RATIO     = 0.10
WEIGHT_DECAY     = 0.01
VAL_RATIO        = 0.10
PATIENCE         = 5

LABEL_SMOOTHING  = 0.10
DROPOUT          = 0.10

LLRD_DECAY       = 0.90

USE_FGM          = True
FGM_EPSILON      = 1.0

USE_RDROP        = False
RDROP_ALPHA      = 4.0

USE_HARD_NEG     = True
MARGIN           = 0.5
LAMBDA_MARGIN    = 0.30

USE_EMA          = True
EMA_DECAY        = 0.999

SEED             = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"=== NAICS-2 Cross-Encoder MAX TOP-1 (seed={SEED}) ===")
print(f"  Model:           {MODEL_NAME}")
print(f"  Max length:      {MAX_LENGTH}")
print(f"  Batch size:      {BATCH_SIZE} queries x {NUM_CLASSES} candidates = {BATCH_SIZE*NUM_CLASSES} seq/step")
print(f"  Epochs:          {NUM_EPOCHS}  (early stop patience {PATIENCE})")
print(f"  LR:              {LEARNING_RATE}  (LLRD decay {LLRD_DECAY})")
print(f"  Dropout:         {DROPOUT}  | Label smoothing: {LABEL_SMOOTHING}")
print(f"  FGM:             {USE_FGM} (eps={FGM_EPSILON})")
print(f"  R-Drop:          {USE_RDROP} (alpha={RDROP_ALPHA})")
print(f"  Hard-neg margin: {USE_HARD_NEG} (m={MARGIN}, lambda={LAMBDA_MARGIN})")
print(f"  EMA weights:     {USE_EMA} (decay={EMA_DECAY})")
print()
print("To train an ensemble, rerun the notebook with SEED=123 and SEED=456,")
print("and run the ensemble cell at the end to average their logits.")

=== NAICS-2 Cross-Encoder MAX TOP-1 (seed=42) ===
  Model:           microsoft/deberta-v3-base
  Max length:      192
  Batch size:      8 queries x 20 candidates = 160 seq/step
  Epochs:          12  (early stop patience 5)
  LR:              1.5e-05  (LLRD decay 0.9)
  Dropout:         0.1  | Label smoothing: 0.1
  FGM:             True (eps=1.0)
  R-Drop:          False (alpha=4.0)
  Hard-neg margin: True (m=0.5, lambda=0.3)
  EMA weights:     True (decay=0.999)

To train an ensemble, rerun the notebook with SEED=123 and SEED=456,
and run the ensemble cell at the end to average their logits.


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=1)
config.hidden_dropout_prob       = DROPOUT
config.attention_probs_dropout_prob = DROPOUT

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=config, torch_dtype=torch.float32
).to(device)

print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Hidden dropout: {config.hidden_dropout_prob}, Attn dropout: {config.attention_probs_dropout_prob}")
print(f"Num encoder layers: {config.num_hidden_layers}")

The tokenizer you are loading from 'microsoft/deberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight        

Total params: 184,422,913
Hidden dropout: 0.1, Attn dropout: 0.1
Num encoder layers: 12


In [7]:
def pretokenize(queries, num_classes, max_length, label_name):
    ids, masks = [], []
    for i, q in enumerate(queries):
        pairs_ids, pairs_masks = [], []
        for c_idx in range(num_classes):
            enc = tokenizer(
                q, corpus_texts[c_idx],
                max_length=max_length, truncation=True,
                padding='max_length', return_tensors='pt'
            )
            pairs_ids.append(enc['input_ids'].squeeze(0))
            pairs_masks.append(enc['attention_mask'].squeeze(0))
        ids.append(torch.stack(pairs_ids))
        masks.append(torch.stack(pairs_masks))
        if (i + 1) % 2000 == 0:
            print(f"  [{label_name}] {i+1}/{len(queries)} done")
    return torch.stack(ids), torch.stack(masks)


print(f"Pre-tokenizing {len(df)} real samples x {NUM_CLASSES} candidates...")
all_input_ids, all_attention_masks = pretokenize(
    df['clean_description'].tolist(), NUM_CLASSES, MAX_LENGTH, "real"
)
all_labels = torch.tensor(df['naics2_idx'].values, dtype=torch.long)
print(f"Real pre-tokenized shape: {all_input_ids.shape}")

print(f"\nPre-tokenizing {len(df_aug)} augmentation samples x {NUM_CLASSES} candidates...")
aug_input_ids, aug_attention_masks = pretokenize(
    df_aug['clean_description'].tolist(), NUM_CLASSES, MAX_LENGTH, "aug"
)
aug_labels = torch.tensor(df_aug['naics2_idx'].values, dtype=torch.long)
print(f"Aug pre-tokenized shape: {aug_input_ids.shape}")

total_mem = (all_input_ids.nbytes + all_attention_masks.nbytes + aug_input_ids.nbytes + aug_attention_masks.nbytes) / 1e9
print(f"\nTotal memory: {total_mem:.2f} GB")


class PreTokenizedDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'label':          self.labels[idx],
        }

print("Pre-tokenization complete.")

Pre-tokenizing 20535 real samples x 20 candidates...
  [real] 2000/20535 done
  [real] 4000/20535 done
  [real] 6000/20535 done
  [real] 8000/20535 done
  [real] 10000/20535 done
  [real] 12000/20535 done
  [real] 14000/20535 done
  [real] 16000/20535 done
  [real] 18000/20535 done
  [real] 20000/20535 done
Real pre-tokenized shape: torch.Size([20535, 20, 192])

Pre-tokenizing 2173 augmentation samples x 20 candidates...
  [aug] 2000/2173 done
Aug pre-tokenized shape: torch.Size([2173, 20, 192])

Total memory: 1.40 GB
Pre-tokenization complete.


In [8]:
train_idx, val_idx = train_test_split(
    np.arange(len(df)), test_size=VAL_RATIO, random_state=SEED, stratify=df['naics2_idx']
)

train_input_ids       = torch.cat([all_input_ids[train_idx], aug_input_ids], dim=0)
train_attention_masks = torch.cat([all_attention_masks[train_idx], aug_attention_masks], dim=0)
train_labels          = torch.cat([all_labels[train_idx], aug_labels], dim=0)

train_dataset = PreTokenizedDataset(train_input_ids, train_attention_masks, train_labels)
val_descriptions = df['clean_description'].iloc[val_idx].tolist()
val_labels       = df['naics2_idx'].iloc[val_idx].tolist()

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)

print(f"Real training samples:  {len(train_idx)}")
print(f"Augmentation samples:   {len(df_aug)}")
print(f"Total training samples: {len(train_dataset)}  ({len(train_loader)} batches)")
print(f"Val samples:            {len(val_descriptions)}  (real only)")

Real training samples:  18481
Augmentation samples:   2173
Total training samples: 20654  (2582 batches)
Val samples:            2054  (real only)


In [9]:
class FGM:
    def __init__(self, model, epsilon=1.0, emb_name='word_embeddings'):
        self.model = model
        self.epsilon = epsilon
        self.emb_name = emb_name
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name:
                if name in self.backup:
                    param.data = self.backup[name]
        self.backup = {}


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name].mul_(self.decay).add_(param.data, alpha=1 - self.decay)

    def apply_shadow(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name].clone()

    def restore(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data = self.backup[name]
        self.backup = {}


def listwise_loss(scores, labels, class_weights, label_smoothing=0.10,
                  use_hard_neg=True, margin=0.5, lambda_margin=0.30):
    ce = F.cross_entropy(scores, labels, weight=class_weights, label_smoothing=label_smoothing)
    if not use_hard_neg:
        return ce, ce, torch.tensor(0.0, device=scores.device)

    pos_scores = scores.gather(1, labels.unsqueeze(1)).squeeze(1)
    neg_scores = scores.clone()
    neg_scores.scatter_(1, labels.unsqueeze(1), float('-inf'))
    hardest_neg = neg_scores.max(dim=1).values
    margin_loss = F.relu(margin - pos_scores + hardest_neg).mean()

    total = ce + lambda_margin * margin_loss
    return total, ce, margin_loss


def r_drop_kl(logits1, logits2):
    log_p = F.log_softmax(logits1, dim=-1)
    log_q = F.log_softmax(logits2, dim=-1)
    p = log_p.exp()
    q = log_q.exp()
    return (F.kl_div(log_p, q, reduction='batchmean') +
            F.kl_div(log_q, p, reduction='batchmean')) / 2.0


def get_llrd_param_groups(model, base_lr, weight_decay, decay_rate=0.9):
    no_decay = ['bias', 'LayerNorm.weight']
    num_layers = model.config.num_hidden_layers

    param_groups = []
    seen = set()

    for n, p in model.named_parameters():
        if 'embeddings' in n and 'rel_embeddings' not in n and id(p) not in seen:
            wd = 0.0 if any(nd in n for nd in no_decay) else weight_decay
            param_groups.append({
                'params': [p],
                'lr': base_lr * (decay_rate ** (num_layers + 1)),
                'weight_decay': wd,
            })
            seen.add(id(p))

    for layer_id in range(num_layers):
        prefix = f'encoder.layer.{layer_id}.'
        for n, p in model.named_parameters():
            if prefix in n and id(p) not in seen:
                wd = 0.0 if any(nd in n for nd in no_decay) else weight_decay
                param_groups.append({
                    'params': [p],
                    'lr': base_lr * (decay_rate ** (num_layers - layer_id - 1)),
                    'weight_decay': wd,
                })
                seen.add(id(p))

    for n, p in model.named_parameters():
        if id(p) not in seen:
            wd = 0.0 if any(nd in n for nd in no_decay) else weight_decay
            param_groups.append({
                'params': [p],
                'lr': base_lr,
                'weight_decay': wd,
            })
            seen.add(id(p))

    print(f"LLRD: {len(param_groups)} param groups")
    print(f"  Min LR: {min(g['lr'] for g in param_groups):.2e}")
    print(f"  Max LR: {max(g['lr'] for g in param_groups):.2e}")
    return param_groups


print("Helper classes defined: FGM, ModelEMA, listwise_loss, r_drop_kl, get_llrd_param_groups")

Helper classes defined: FGM, ModelEMA, listwise_loss, r_drop_kl, get_llrd_param_groups


In [10]:
@torch.no_grad()
def evaluate(model, tokenizer, descriptions, labels, corpus_texts, max_length, score_batch=64):
    model.eval()
    n = len(descriptions)
    num_classes = len(corpus_texts)
    top1, top3, top5 = 0, 0, 0
    all_preds = []

    for i in range(n):
        query = descriptions[i]
        true_label = labels[i]
        scores = []

        for j in range(0, num_classes, score_batch):
            batch_docs = corpus_texts[j:j+score_batch]
            enc = tokenizer(
                [query] * len(batch_docs), batch_docs,
                max_length=max_length, truncation=True, padding=True,
                return_tensors='pt'
            )
            enc = {k: v.to(device) for k, v in enc.items() if k in ['input_ids', 'attention_mask']}
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits = model(**enc).logits.squeeze(-1)
            scores.extend(logits.float().cpu().tolist())

        scores_t = torch.tensor(scores)
        topk = scores_t.topk(min(5, num_classes)).indices.tolist()
        all_preds.append(topk[0])

        if topk[0] == true_label: top1 += 1
        if true_label in topk[:3]: top3 += 1
        if true_label in topk[:5]: top5 += 1

    macro_f1    = f1_score(labels, all_preds, average='macro',    zero_division=0)
    weighted_f1 = f1_score(labels, all_preds, average='weighted', zero_division=0)
    return {
        'top1': top1 / n, 'top3': top3 / n, 'top5': top5 / n,
        'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
        'all_preds': all_preds,
    }


print("Evaluation function defined.")

Evaluation function defined.


In [11]:
from transformers import get_cosine_schedule_with_warmup
from tqdm.auto import tqdm
import time

CHECKPOINT_DIR = f"/content/drive/MyDrive/naics2_base_curated_checkpoints_seed{SEED}"
RESULTS_DIR    = f"results_naics2_base_curated_seed{SEED}"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

label_counts = np.bincount(train_labels.numpy(), minlength=NUM_CLASSES).astype(np.float64)
class_weights_np = 1.0 / np.sqrt(np.maximum(label_counts, 1.0))
class_weights_np = class_weights_np / class_weights_np.sum() * NUM_CLASSES
class_weights = torch.tensor(class_weights_np, dtype=torch.float, device=device)

print("Class weights (sqrt inv-freq, normalized):")
for code, title, cnt, w in zip(sector_codes, sector_titles, label_counts, class_weights_np):
    print(f"  {code:>5} ({int(cnt):>5})  weight={w:.3f}  {title[:50]}")

param_groups = get_llrd_param_groups(model, LEARNING_RATE, WEIGHT_DECAY, LLRD_DECAY)
optimizer = torch.optim.AdamW(param_groups)

total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

fgm = FGM(model, epsilon=FGM_EPSILON, emb_name='word_embeddings') if USE_FGM else None
ema = ModelEMA(model, decay=EMA_DECAY) if USE_EMA else None

print(f"\nTotal steps: {total_steps}  |  Warmup: {warmup_steps}")

best_top1 = 0.0
best_epoch = 0
patience_counter = 0
epoch_log = []

CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "checkpoint.pt")
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model2.pt")

start_epoch = 0
if os.path.exists(CHECKPOINT_PATH):
    print(f"\nResuming from checkpoint: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    if USE_EMA and 'ema' in ckpt:
        ema.shadow = {k: v.to(device) for k, v in ckpt['ema'].items()}
    start_epoch = ckpt['epoch']
    best_top1 = ckpt.get('best_top1', 0.0)
    best_epoch = ckpt.get('best_epoch', 0)
    epoch_log = ckpt.get('epoch_log', [])
    patience_counter = ckpt.get('patience_counter', 0)
    print(f"Resumed at epoch {start_epoch}, best Top-1 so far: {best_top1:.4f}")

print(f"\nTraining epochs {start_epoch+1} to {NUM_EPOCHS}")
print("=" * 100)
print(f" Ep   TrLoss   CE     Marg   KL      Top1    Top3    Top5   MacF1    WtF1         LR")
print("=" * 100)

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    total_loss, total_ce, total_marg, total_kl = 0.0, 0.0, 0.0, 0.0
    nb = 0
    epoch_start = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
    for batch in pbar:
        ids   = batch['input_ids'].to(device)
        masks = batch['attention_mask'].to(device)
        lbls  = batch['label'].to(device)
        B, K, L = ids.shape
        flat_ids   = ids.view(B*K, L)
        flat_masks = masks.view(B*K, L)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            logits1 = model(input_ids=flat_ids, attention_mask=flat_masks).logits.squeeze(-1).view(B, K).float()

            if USE_RDROP:
                logits2 = model(input_ids=flat_ids, attention_mask=flat_masks).logits.squeeze(-1).view(B, K).float()
                loss1, ce1, marg1 = listwise_loss(logits1, lbls, class_weights, LABEL_SMOOTHING,
                                                  USE_HARD_NEG, MARGIN, LAMBDA_MARGIN)
                loss2, ce2, marg2 = listwise_loss(logits2, lbls, class_weights, LABEL_SMOOTHING,
                                                  USE_HARD_NEG, MARGIN, LAMBDA_MARGIN)
                kl = r_drop_kl(logits1, logits2)
                loss = (loss1 + loss2) / 2 + RDROP_ALPHA * kl
                total_kl += kl.item()
                total_ce += (ce1.item() + ce2.item()) / 2
                total_marg += (marg1.item() + marg2.item()) / 2
            else:
                loss, ce1, marg1 = listwise_loss(logits1, lbls, class_weights, LABEL_SMOOTHING,
                                                 USE_HARD_NEG, MARGIN, LAMBDA_MARGIN)
                total_ce += ce1.item()
                total_marg += marg1.item()

        loss.backward()

        if USE_FGM:
            fgm.attack()
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                logits_adv = model(input_ids=flat_ids, attention_mask=flat_masks).logits.squeeze(-1).view(B, K).float()
                loss_adv, _, _ = listwise_loss(logits_adv, lbls, class_weights, LABEL_SMOOTHING,
                                               USE_HARD_NEG, MARGIN, LAMBDA_MARGIN)
            loss_adv.backward()
            fgm.restore()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        if USE_EMA:
            ema.update(model)

        total_loss += loss.item()
        nb += 1
        if nb % 25 == 0:
            pbar.set_postfix(loss=f"{total_loss/nb:.3f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    pbar.close()
    epoch_time = time.time() - epoch_start
    print(f"  Epoch {epoch+1} train time: {epoch_time/60:.1f} min")

    avg_loss = total_loss / nb
    avg_ce   = total_ce / nb
    avg_marg = total_marg / nb
    avg_kl   = total_kl / nb if USE_RDROP else 0.0

    if USE_EMA:
        ema.apply_shadow(model)
    eval_results = evaluate(model, tokenizer, val_descriptions, val_labels, corpus_texts, MAX_LENGTH)
    if USE_EMA:
        ema.restore(model)

    cur_lr = optimizer.param_groups[0]['lr']
    log = {
        'epoch': epoch + 1, 'train_loss': avg_loss, 'ce_loss': avg_ce,
        'margin_loss': avg_marg, 'kl_loss': avg_kl, 'lr': cur_lr,
        **{k: v for k, v in eval_results.items() if k != 'all_preds'}
    }
    epoch_log.append(log)
    print(f"  {epoch+1:>2}  {avg_loss:6.4f}  {avg_ce:5.3f}  {avg_marg:5.3f}  {avg_kl:5.3f}  "
          f"{eval_results['top1']:.4f}  {eval_results['top3']:.4f}  {eval_results['top5']:.4f}  "
          f"{eval_results['macro_f1']:.4f}  {eval_results['weighted_f1']:.4f}   {cur_lr:.2e}")

    is_best = eval_results['top1'] > best_top1
    if is_best:
        best_top1 = eval_results['top1']
        best_epoch = epoch + 1
        patience_counter = 0
        if USE_EMA:
            save_state = {}
            for k, v in model.state_dict().items():
                if k in ema.shadow:
                    save_state[k] = ema.shadow[k].detach().cpu().clone()
                else:
                    save_state[k] = v.detach().cpu().clone()
        else:
            save_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save(save_state, BEST_MODEL_PATH)
    else:
        patience_counter += 1

    torch.save({
        'epoch': epoch + 1,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'ema': {k: v.cpu() for k, v in ema.shadow.items()} if USE_EMA else None,
        'best_top1': best_top1,
        'best_epoch': best_epoch,
        'epoch_log': epoch_log,
        'patience_counter': patience_counter,
    }, CHECKPOINT_PATH)

    pd.DataFrame(epoch_log).to_csv(os.path.join(RESULTS_DIR, "epoch_log.csv"), index=False)

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\n{'=' * 100}")
print(f"Best Top-1: {best_top1:.4f} at epoch {best_epoch}")
print(f"Best model saved to: {BEST_MODEL_PATH}")

Class weights (sqrt inv-freq, normalized):
     11 (  696)  weight=0.848  Agriculture, Forestry, Fishing and Hunting
     21 (  465)  weight=1.038  Mining, Quarrying, and Oil and Gas Extraction
     22 (  154)  weight=1.803  Utilities
     23 (  737)  weight=0.824  Construction
  31-33 ( 7335)  weight=0.261  Manufacturing
     42 ( 1944)  weight=0.507  Wholesale Trade
  44-45 ( 1703)  weight=0.542  Retail Trade
  48-49 (  916)  weight=0.739  Transportation and Warehousing
     51 (  715)  weight=0.837  Information
     52 (  797)  weight=0.793  Finance and Insurance
     53 (  524)  weight=0.977  Real Estate and Rental and Leasing
     54 ( 1045)  weight=0.692  Professional, Scientific, and Technical Services
     55 (   68)  weight=2.713  Management of Companies and Enterprises
     56 (  693)  weight=0.850  Administrative and Support and Waste Management an
     61 (  269)  weight=1.364  Educational Services
     62 (  721)  weight=0.833  Health Care and Social Assistance
     71 (  

Epoch 1/12:   0%|          | 0/2582 [00:00<?, ?it/s]

  Epoch 1 train time: 33.0 min
   1  3.3190  3.129  0.635  0.000  0.3559  0.5472  0.6470  0.0772  0.2686   3.18e-06


Epoch 2/12:   0%|          | 0/2582 [00:00<?, ?it/s]

  Epoch 2 train time: 32.9 min
   2  2.6462  2.415  0.771  0.000  0.5837  0.7911  0.8627  0.4569  0.5598   3.76e-06


...

  Epoch 6 train time: 33.0 min
   6  2.1285  1.929  0.665  0.000  0.5623  0.7795  0.8681  0.4686  0.5373   2.24e-06


Epoch 7/12:   0%|          | 0/2582 [00:00<?, ?it/s]

  Epoch 7 train time: 33.0 min
   7  2.1112  1.913  0.661  0.000  0.2833  0.4966  0.6456  0.3216  0.2585   1.69e-06


Epoch 8/12:   0%|          | 0/2582 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
print("Reloading EMA-best weights for final evaluation...")
state = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(state)

final = evaluate(model, tokenizer, val_descriptions, val_labels, corpus_texts, MAX_LENGTH)
print(f"\n=== FINAL VALIDATION (seed={SEED}) ===")
print(f"  Top-1: {final['top1']:.4f}")
print(f"  Top-3: {final['top3']:.4f}")
print(f"  Top-5: {final['top5']:.4f}")
print(f"  Macro F1: {final['macro_f1']:.4f}")
print(f"  Weighted F1: {final['weighted_f1']:.4f}")

results = {
    'seed': SEED,
    'best_top1': best_top1,
    'best_epoch': best_epoch,
    'final': {k: v for k, v in final.items() if k != 'all_preds'},
    'config': {
        'MAX_LENGTH': MAX_LENGTH, 'BATCH_SIZE': BATCH_SIZE, 'NUM_EPOCHS': NUM_EPOCHS,
        'LEARNING_RATE': LEARNING_RATE, 'LABEL_SMOOTHING': LABEL_SMOOTHING,
        'DROPOUT': DROPOUT, 'LLRD_DECAY': LLRD_DECAY,
        'USE_FGM': USE_FGM, 'FGM_EPSILON': FGM_EPSILON,
        'USE_RDROP': USE_RDROP, 'RDROP_ALPHA': RDROP_ALPHA,
        'USE_HARD_NEG': USE_HARD_NEG, 'MARGIN': MARGIN, 'LAMBDA_MARGIN': LAMBDA_MARGIN,
        'USE_EMA': USE_EMA, 'EMA_DECAY': EMA_DECAY,
    },
    'epoch_log': epoch_log,
}
with open(os.path.join(RESULTS_DIR, "results.json"), 'w') as f:
    json.dump(results, f, indent=2)

shutil.copy(BEST_MODEL_PATH, os.path.join(RESULTS_DIR, "best_model2.pt"))
zip_path = shutil.make_archive(f"naics2_base_curated_seed{SEED}", 'zip', RESULTS_DIR)
print(f"\nResults archived: {zip_path}")
print(f"Contents of {RESULTS_DIR}:")
for f in os.listdir(RESULTS_DIR):
    size_mb = os.path.getsize(os.path.join(RESULTS_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)
    print(f"\nDownload triggered for: {zip_path}")
except ImportError:
    print(f"\nNot running in Colab - results at: {os.path.abspath(zip_path)}")

## Ensemble inference across 3 seeds

In [ ]:
ENSEMBLE_SEEDS = [42, 123, 456]
ckpt_paths = []
for s in ENSEMBLE_SEEDS:
    p = f"/content/drive/MyDrive/naics2_llrd_adv_hn_ens_checkpoints_seed{s}/best_model2.pt"
    if os.path.exists(p):
        ckpt_paths.append((s, p))
        print(f"Found seed {s} checkpoint")
    else:
        print(f"Missing seed {s} checkpoint -> skipping")

if len(ckpt_paths) < 2:
    print("\nNeed at least 2 trained seeds to ensemble. Train more seeds first.")
else:
    print(f"\nEnsembling {len(ckpt_paths)} models...")
    n = len(val_descriptions)
    accumulated_scores = np.zeros((n, NUM_CLASSES), dtype=np.float64)

    for s, path in ckpt_paths:
        print(f"  Scoring with seed={s}...")
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()
        with torch.no_grad():
            for i in range(n):
                query = val_descriptions[i]
                scores = []
                for j in range(0, NUM_CLASSES, 64):
                    batch_docs = corpus_texts[j:j+64]
                    enc = tokenizer([query] * len(batch_docs), batch_docs,
                                    max_length=MAX_LENGTH, truncation=True, padding=True,
                                    return_tensors='pt')
                    enc = {k: v.to(device) for k, v in enc.items() if k in ['input_ids', 'attention_mask']}
                    logits = model(**enc).logits.squeeze(-1)
                    scores.extend(logits.cpu().tolist())
                accumulated_scores[i] += np.array(scores)

    avg_scores = accumulated_scores / len(ckpt_paths)
    top1, top3, top5 = 0, 0, 0
    all_preds = []
    for i in range(n):
        ranked = np.argsort(-avg_scores[i])[:5].tolist()
        all_preds.append(ranked[0])
        if ranked[0] == val_labels[i]: top1 += 1
        if val_labels[i] in ranked[:3]: top3 += 1
        if val_labels[i] in ranked[:5]: top5 += 1

    macro_f1    = f1_score(val_labels, all_preds, average='macro',    zero_division=0)
    weighted_f1 = f1_score(val_labels, all_preds, average='weighted', zero_division=0)

    print(f"\n=== ENSEMBLE ({len(ckpt_paths)} seeds) ===")
    print(f"  Top-1: {top1/n:.4f}")
    print(f"  Top-3: {top3/n:.4f}")
    print(f"  Top-5: {top5/n:.4f}")
    print(f"  Macro F1: {macro_f1:.4f}")
    print(f"  Weighted F1: {weighted_f1:.4f}")

    ens_results = {
        'seeds': [s for s, _ in ckpt_paths],
        'top1': top1/n, 'top3': top3/n, 'top5': top5/n,
        'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
    }
    os.makedirs("results_ensemble", exist_ok=True)
    with open("results_ensemble/ensemble_results.json", 'w') as f:
        json.dump(ens_results, f, indent=2)
    print("\nSaved: results_ensemble/ensemble_results.json")